# Fase 4: Serving, Inferencia End-to-End y Evaluación de Recomendaciones
## Premier League Match Recommender

Este notebook documenta la **Fase 4 del sistema**, donde se orquestan todos los componentes desarrollados para servir recomendaciones en producción:
1. **Pipeline de Inferencia End-to-End**: Evaluación simultánea de la jornada mediante el orquestador central `MatchScorer`.
2. **Generación del Top 3 de Partidos Recomendados**: Selección de encuentros con mayor potencial de espectáculo (*Excitement Score*) y explicabilidad contextual estructurada con `SemanticExplainer`.
3. **Análisis de Sensibilidad por Perfiles de Usuario**: Benchmarking y comparación de rankings bajo diferentes estilos de aficionado (`cv_optimal`, `balanced`, `goal_fest`, `tension`, `fan_hype`).
4. **Comparativa Metodológica**: Modelo Canónico Ponderado (`weighted`) vs Modelo Supervisado de Machine Learning (`ml_hybrid`).
5. **Simulación de Endpoints de la API REST (FastAPI)**: Validación de consumo, filtros por fecha (`YYYY-MM-DD`) y esquemas Pydantic serializables.

In [1]:
import sys
import os
from pathlib import Path
import json
import pandas as pd
import numpy as np

BASE_DIR = Path(os.path.abspath("..")) if Path("..").resolve().name == "sistema_de_recomendaciones" else Path(os.path.abspath("."))
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

from src.core.config import settings
from src.core.schemas import RecommendationRequest, RecommendationResponse
from src.data_ingestion.local_provider import LocalDataProvider
from src.features.numerical import NumericalFeatureExtractor
from src.features.nlp import NLPFeatureExtractor
from src.engine.explainer import SemanticExplainer
from src.engine.hybrid_model import HybridMLScorer
from src.engine.scorer import MatchScorer

print("✔ Componentes de Serving y Recomendación listos.")
print(f"✔ Perfiles de usuario disponibles: {list(settings.weight_profiles.keys()) + ['cv_optimal']}")

✔ Componentes de Serving y Recomendación listos.
✔ Perfiles de usuario disponibles: ['balanced', 'cv_optimal', 'goal_fest', 'tension', 'fan_hype', 'cv_optimal']

---
## 1. Recomendación del Top 3 de Partidos (Jornada 29) con Perfil Óptimo (`cv_optimal`)

Ejecutamos el recomendador sobre la jornada 29 utilizando los pesos matemáticamente calibrados por validación cruzada ($k=5$).
Para cada partido recomendado, el sistema genera de forma automática justificaciones semánticas estructuradas que explican *por qué* merece ser visto.

In [2]:
local_provider = LocalDataProvider(data_dir=settings.data_dir)
scorer = MatchScorer(data_provider=local_provider)

# Petición para el Top 3 con perfil 'cv_optimal'
request_optimal = RecommendationRequest(
    gameweek=29,
    profile="cv_optimal",
    top_n=3,
    model_type="weighted"
)

response_optimal = scorer.recommend(request_optimal)

print("=" * 75)
print(f" ⚽ PREMIER LEAGUE MATCH RECOMMENDER — TOP {response_optimal.total_evaluated_matches} JORNADA {response_optimal.gameweek}")
print(f" ⚙️  Perfil Utilizado: '{response_optimal.profile_used.upper()}' | Modo: '{response_optimal.scoring_mode}'")
print("=" * 75)

for match in response_optimal.top_matches:
    home = match.fixture.home_team
    away = match.fixture.away_team
    num = match.numerical_features
    nlp = match.nlp_features
    att = match.feature_attributions
    
    print(f"\n🏆 RANK #{match.rank} | {home.name} (#{home.position}) vs {away.name} (#{away.position})")
    print(f"   ⭐ Excitement Score: {match.excitement_score} / 100")
    print(f"   🎯 Choque Táctico: {num.tactical_style_label} (Intensidad: {int(num.tactical_clash_intensity*100)}%)")
    print(f"   📊 Métricas Clave:")
    print(f"      • xG Combinado: {num.xg_combined} | Paridad: {int(num.table_parity*100)}% | Star Power: {num.star_power}")
    print(f"      • Sentimiento Afición: {nlp.fan_excitement_index}/100 | Velocidad Prensa: {nlp.media_velocity}/100")
    print(f"   📈 Desglose de Factores:")
    print(f"      • Goles (xG): {att.get('xg_offensive_potential', 0)}% | Tabla: {att.get('competitive_stakes_parity', 0)}% | Afición: {att.get('fan_hype_and_drama', 0)}% | Prensa: {att.get('media_coverage_and_stars', 0)}%")
    print(f"   💡 Justificación Semántica:")
    for r in match.reasons:
        print(f"      - [{r.category}] {r.title}: {r.description}")

print("=" * 75)

 ⚽ PREMIER LEAGUE MATCH RECOMMENDER — TOP 10 JORNADA 29
 ⚙️  Perfil Utilizado: 'CV_OPTIMAL' | Modo: 'weighted'

🏆 RANK #1 | Manchester City (#3) vs Arsenal (#1)
   ⭐ Excitement Score: 82.25 / 100
   🎯 Choque Táctico: Guerra Ofensiva Abierta (Ataque vs Ataque) (Intensidad: 91%)
   📊 Métricas Clave:
      • xG Combinado: 4.75 | Paridad: 90% | Star Power: 650.5
      • Sentimiento Afición: 64.19/100 | Velocidad Prensa: 100.0/100
   📈 Desglose de Factores:
      • Goles (xG): 22.6% | Tabla: 21.9% | Afición: 31.2% | Prensa: 24.3%
   💡 Justificación Semántica:
      - [CHOQUE TÁCTICO] Guerra Ofensiva Abierta (Ataque vs Ataque) (91% de intensidad): Duelo de alta fricción táctica: se proyectan 131 pases progresivos y 281 presiones defensivas acumuladas entre ambos conjuntos.
      - [DRAMATISMO HISTÓRICO] Alta propensión a desenlace épico (70% de dramatismo): El historial narrativo y análisis táctico anticipan un duelo volátil con alta probabilidad de desenlaces dramáticos, goles tardíos o alt

---
## 2. Análisis de Sensibilidad y Comparativa por Perfiles de Aficionado

Diferentes aficionados tienen preferencias distintas:
- `cv_optimal`: Máxima precisión de ranking calibrada por machine learning y validación cruzada.
- `goal_fest`: Prioriza partidos con alta probabilidad de goles y llegadas frecuentes ($w_{xG} = 0.55$).
- `tension`: Prioriza partidos cerrados, derbis y duelos directos por puestos clave ($w_{Paridad} = 0.45$).
- `fan_hype`: Prioriza encuentros con debate viral masivo en Reddit y redes sociales ($w_{Fan} = 0.50$).

Evaluamos cómo varía el orden de recomendación según el perfil seleccionado.

In [3]:
profiles = ["cv_optimal", "balanced", "goal_fest", "tension", "fan_hype"]
profile_rankings = {}

for p in profiles:
    req = RecommendationRequest(gameweek=29, profile=p, top_n=5, model_type="weighted")
    res = scorer.recommend(req)
    profile_rankings[p] = [
        f"#{m.rank} {m.fixture.home_team.short_name} vs {m.fixture.away_team.short_name} ({m.excitement_score})"
        for m in res.top_matches
    ]

df_profiles = pd.DataFrame(profile_rankings)
df_profiles.index = [f"Top {i+1}" for i in range(5)]
print("--- COMPARATIVA DE RANKINGS TOP 5 SEGÚN PERFIL DE AFICIONADO ---")
print(df_profiles.to_string())

--- COMPARATIVA DE RANKINGS TOP 5 SEGÚN PERFIL DE AFICIONADO ---
                  cv_optimal               balanced              goal_fest                tension               fan_hype
Top 1  #1 MCI vs ARS (82.25)  #1 MCI vs ARS (86.05)  #1 MCI vs ARS (87.55)  #1 MCI vs ARS (85.48)  #1 MCI vs ARS (80.46)
Top 2  #2 TOT vs CHE (65.82)  #2 TOT vs CHE (67.81)  #2 TOT vs CHE (68.62)  #2 TOT vs CHE (67.53)  #2 TOT vs CHE (64.95)
Top 3  #3 AVL vs WOL (61.52)  #3 AVL vs WOL (60.07)  #3 LIV vs EVE (61.63)  #3 AVL vs WOL (65.07)  #3 AVL vs WOL (63.08)
Top 4  #4 LIV vs EVE (57.55)  #4 NEW vs WHU (57.63)  #4 AVL vs WOL (55.76)  #4 NEW vs WHU (63.77)  #4 LIV vs EVE (56.22)
Top 5  #5 NEW vs WHU (53.93)  #5 LIV vs EVE (56.55)  #5 NEW vs WHU (54.11)  #5 LUT vs NFO (60.54)  #5 NEW vs WHU (53.94)

---
## 3. Comparativa Metodológica: Ponderado Canónico vs Machine Learning Multimodal (`ml_hybrid`)

Comparamos los resultados de evaluación entre:
1. **Modo Canónico Ponderado (`weighted`)**: Combinación lineal basada en pesos normalizados.
2. **Modo Machine Learning Supervisado (`ml_hybrid`)**: Modelo no lineal `GradientBoostingRegressor` que evalúa las 12 características multimodales conjuntas.

In [4]:
req_weighted = RecommendationRequest(gameweek=29, profile="cv_optimal", top_n=10, model_type="weighted")
req_ml = RecommendationRequest(gameweek=29, profile="cv_optimal", top_n=10, model_type="ml_hybrid")

res_weighted = scorer.recommend(req_weighted)
res_ml = scorer.recommend(req_ml)

comparison_data = []
ml_scores_by_id = {m.fixture.id: m for m in res_ml.top_matches}

for m_w in res_weighted.top_matches:
    m_ml = ml_scores_by_id.get(m_w.fixture.id)
    comparison_data.append({
        "fixture_id": m_w.fixture.id,
        "match": f"{m_w.fixture.home_team.short_name} vs {m_w.fixture.away_team.short_name}",
        "score_weighted": m_w.excitement_score,
        "rank_weighted": m_w.rank,
        "score_ml": m_ml.excitement_score if m_ml else 0.0,
        "rank_ml": m_ml.rank if m_ml else 0,
        "diff_score": round((m_ml.excitement_score - m_w.excitement_score) if m_ml else 0.0, 2)
    })

df_comp = pd.DataFrame(comparison_data).sort_values("rank_weighted").reset_index(drop=True)
print("--- COMPARACIÓN: MODELO PONDERADO vs GRADIENT BOOSTING ML ---")
print(df_comp[["match", "score_weighted", "rank_weighted", "score_ml", "rank_ml", "diff_score"]].to_string(index=False))

--- COMPARACIÓN: MODELO PONDERADO vs GRADIENT BOOSTING ML ---
     match  score_weighted  rank_weighted  score_ml  rank_ml  diff_score
MCI vs ARS           82.25              1     89.01        1        6.76
TOT vs CHE           65.82              2     69.91        2        4.09
AVL vs WOL           61.52              3     68.17        3        6.65
LIV vs EVE           57.55              4     66.98        4        9.43
NEW vs WHU           53.93              5     56.03        7        2.10
BHA vs CRY           51.11              6     65.13        5       14.02
MUN vs FUL           47.75              7     58.97        6       11.22
LUT vs NFO           44.77              8     49.24        9        4.47
BOU vs SHU           44.36              9     54.15        8        9.79
BUR vs BRE           36.22             10     47.55       10       11.33

---
## 4. Simulación de Endpoints de la API REST (FastAPI) y Filtrado por Fecha

La API REST implementa endpoints diseñados para clientes web, móviles y dashboards:
- `GET /api/v1/recommendations/top3?gameweek=29&date=2026-03-31`: Filtrado específico de partidos para un día en particular.
- `GET /api/v1/fixtures/{id}/analysis`: Auditoría técnica desglosada para un encuentro.
- `GET /api/v1/hybrid/cross-validation`: Telemetría de validación cruzada y pesos calibrados.

Simulamos estas consultas programáticamente y validamos su serialización completa.

In [5]:
# 1. Simulación de recomendación con filtro de fecha (Fase 4)
req_date = RecommendationRequest(
    gameweek=29,
    date="2026-03-31",
    profile="cv_optimal",
    top_n=2
)
res_date = scorer.recommend(req_date)

print(f"--- CONSULTA CON FILTRO DE FECHA ({res_date.date_filter}) ---")
print(f"Partidos programados para esa fecha: {res_date.total_evaluated_matches}")
for m in res_date.top_matches:
    print(f" • #{m.rank} {m.fixture.home_team.name} vs {m.fixture.away_team.name} (Kickoff: {m.fixture.kickoff_time}) -> Score: {m.excitement_score}")

# 2. Simulación de análisis individual desglosado (/fixtures/{id}/analysis)
target_fixture_id = 101 # Man City vs Arsenal
target_fixture = next(f for f in local_provider.get_fixtures() if f.id == target_fixture_id)
weights_optimal = scorer.ml_scorer.optimizer.get_result().optimal_weights
analysis_match = scorer.score_match(target_fixture, weights_optimal, scoring_mode="ml_hybrid")

json_payload = json.loads(analysis_match.model_dump_json())
print(f"\n--- TELEMETRÍA JSON SERIALIZADA PARA PARTIDO {target_fixture_id} ({analysis_match.fixture.home_team.short_name} vs {analysis_match.fixture.away_team.short_name}) ---")
print(f" • Excitement Score: {json_payload['excitement_score']}")
print(f" • Choque Táctico: {json_payload['numerical_features']['tactical_style_label']}")
print(f" • Probabilidad de Drama: {json_payload['nlp_features']['dramatic_intensity_score']}")
print(f" • Atribuciones: {json_payload['feature_attributions']}")
print(f" • Justificaciones generadas: {len(json_payload['reasons'])} razones semánticas")

print("\n✅ Fase 4 completada: Pipeline de recomendación, serving y API REST validados.")

--- CONSULTA CON FILTRO DE FECHA (2026-03-31) ---
Partidos programados para esa fecha: 2
 • #1 Manchester City vs Arsenal (Kickoff: 2026-03-31T15:30:00Z) -> Score: 82.25
 • #2 Brighton & Hove Albion vs Crystal Palace (Kickoff: 2026-03-31T13:00:00Z) -> Score: 51.11

--- TELEMETRÍA JSON SERIALIZADA PARA PARTIDO 101 (MCI vs ARS) ---
 • Excitement Score: 89.01
 • Choque Táctico: Guerra Ofensiva Abierta (Ataque vs Ataque)
 • Probabilidad de Drama: 0.704
 • Atribuciones: {'xg_offensive_potential': 18.6, 'competitive_stakes_parity': 27.7, 'fan_hype_and_drama': 17.8, 'media_coverage_and_stars': 35.8}
 • Justificaciones generadas: 6 razones semánticas

✅ Fase 4 completada: Pipeline de recomendación, serving y API REST validados.